# Extraction validation + calibration (Layer 2 — run once, offline)

**This is the only place hand-labels are read.** It takes a *frozen* hand-labelled sample,
measures how good the label-free extractor is, and fits a **calibration mapping** so that a
confidence of 0.8 means ~80% correct on **unseen** cases. That unseen-data claim is validated
**out-of-fold** (stratified 5-fold CV) and quantified as **ECE**, so the calibration is never
scored on rows it was fitted on.

It produces:
- per-field **exact-match accuracy** for the metadata tier (any `gold_*` column found),
- **precision / recall / F1** for `alienation_alleged`,
- a **reliability diagram** (raw + out-of-fold calibrated bins) → `figures/`,
- **ECE** for raw and OOF-calibrated confidence,
- a deployed **isotonic calibration** raw→calibrated (refit on all rows) → `data/alienation_calibration.joblib`,
- a metrics **report** (incl. labelling-protocol limitations) → `reports/`.

Run it **once** after labelling. Re-run only to extend the validated domain.

## 1. Configuration + load the frozen labelled sample (graceful if absent)

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, json

DATA_DIR   = Path("../data")
FIG_DIR    = Path("../figures")
REPORT_DIR = Path("../reports")
for d in (FIG_DIR, REPORT_DIR):
    d.mkdir(exist_ok=True)

TABLE_FILE   = DATA_DIR / "echr_extracted.parquet"
LABELED_FILE = DATA_DIR / "echr_labeled_sample.csv"          # <-- your frozen hand-labels
CALIB_OUT    = DATA_DIR / "alienation_calibration.joblib"
FIG_OUT      = FIG_DIR / "alienation_reliability_diagram.png"
REPORT_OUT   = REPORT_DIR / "extraction_validation_report.md"

ALLEGED_THRESHOLD = 0.50


def _load_table():
    if TABLE_FILE.exists():
        return pd.read_parquet(TABLE_FILE)
    csv = TABLE_FILE.with_suffix(".csv")
    return pd.read_csv(csv) if csv.exists() else None


table = _load_table()
labels = pd.read_csv(LABELED_FILE) if LABELED_FILE.exists() else None

READY = table is not None and labels is not None
if table is None:
    print("!! extracted table missing — run echr_extraction.ipynb first.")
if labels is None:
    print(f"!! labelled sample missing: {LABELED_FILE}")
    print("   1) open data/echr_label_template.csv")
    print("   2) fill gold_alienation_alleged (1/0) and confirm gold_outcome")
    print(f"   3) save it as {LABELED_FILE.name}, then re-run this notebook.")
if READY:
    print(f"table: {len(table)} rows | labelled sample: {len(labels)} rows")

table: 1116 rows | labelled sample: 120 rows


## 2. Join labels to the extractor output by `id`
The frozen sample is joined to the live extraction table by `id`, so the metrics are computed
against the *current* extractor, not a stale copy baked into the label file.

In [ ]:
def _to01(s):
    return (pd.Series(s).astype(str).str.strip().str.lower()
            .map({"1": 1, "0": 0, "true": 1, "false": 0, "yes": 1, "no": 0,
                  "y": 1, "n": 0, "t": 1, "f": 0}))


if READY:
    cols = ["id", "respondent_state", "articles", "judgment_date", "importance",
            "genre", "outcome", "outcome_conf", "alienation_alleged", "alienation_conf"]
    ev = labels.merge(table[cols], on="id", how="inner", suffixes=("", "_tbl"))
    ev["gold_alienation"] = _to01(ev["gold_alienation_alleged"])
    ev = ev[ev["gold_alienation"].notna()].copy()
    ev["gold_alienation"] = ev["gold_alienation"].astype(int)
    print(f"joined {len(ev)} labelled rows with a usable gold_alienation value")
    print("gold positive rate in sample:", round(ev.gold_alienation.mean(), 3))
else:
    ev = None

joined 120 labelled rows with a usable gold_alienation value
gold positive rate in sample: 0.333


## 3. Metadata tier — per-field exact-match accuracy
Computed for every `gold_*` column present in the labelled file. Verbatim-copied HUDOC fields
(`respondent_state`, `articles`, `importance`) are exact by construction (provenance = direct
copy); the meaningful test is the **derived** `outcome` field, so the template ships a
`gold_outcome` column. Add more `gold_<field>` columns to deepen this without code changes.

In [ ]:
if ev is not None:
    print("metadata-tier exact-match accuracy:")
    meta_acc = {}
    for gcol in [c for c in ev.columns if c.startswith("gold_") and c != "gold_alienation_alleged"]:
        field = gcol[len("gold_"):]
        if field not in ev.columns:
            continue
        m = ev[gcol].notna() & (ev[gcol].astype(str).str.strip() != "")
        if m.sum() == 0:
            continue
        a = ev.loc[m, field].astype(str).str.strip().str.lower()
        b = ev.loc[m, gcol].astype(str).str.strip().str.lower()
        acc = float((a.values == b.values).mean())
        meta_acc[field] = (acc, int(m.sum()))
        print(f"  {field:16} acc={acc:.3f}  (n={int(m.sum())})")
    # integrity note for the verbatim fields (exact by construction)
    print("\nverbatim-field integrity (non-null coverage; exact-match=1.0 by construction):")
    for f in ["respondent_state", "articles", "importance"]:
        print(f"  {f:16} coverage={ev[f].notna().mean():.3f}")

metadata-tier exact-match accuracy:
  outcome          acc=1.000  (n=95)

verbatim-field integrity (non-null coverage; exact-match=1.0 by construction):
  respondent_state coverage=1.000
  articles         coverage=1.000
  importance       coverage=1.000


## 4. Content tier — precision / recall / F1 for `alienation_alleged`

In [ ]:
if ev is not None:
    from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
    y_true = ev.gold_alienation.to_numpy()
    y_pred = ev.alienation_alleged.astype(int).to_numpy()
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary",
                                                  zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    print(f"alienation_alleged @ threshold {ALLEGED_THRESHOLD}")
    print(f"  precision={p:.3f}  recall={r:.3f}  F1={f1:.3f}  (n={len(ev)})")
    print(f"  confusion: TP={tp} FP={fp} FN={fn} TN={tn}")
    content_metrics = dict(precision=round(p, 3), recall=round(r, 3), f1=round(f1, 3),
                           tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn), n=int(len(ev)))

alienation_alleged @ threshold 0.5
  precision=0.543  recall=0.475  F1=0.507  (n=120)
  confusion: TP=19 FP=16 FN=21 TN=64


## 5. Reliability diagram + cross-validated isotonic calibration + ECE

The claim "a calibrated 0.8 means ~80% correct on **unseen** cases" is only honest if the
calibration is evaluated on rows it was **not fitted on**. So:

1. **Out-of-fold (OOF) calibration** — stratified 5-fold CV: isotonic regression is fitted on
   4 folds and predicts the held-out fold, so every row gets a calibrated confidence from a
   model that never saw its label.
2. **ECE (Expected Calibration Error)** — equal-width bins, weighted mean |observed − predicted|,
   reported for the *raw* confidence and the *OOF-calibrated* confidence. The calibrated ECE is
   the number the thesis can cite as an unseen-data claim.
3. **Deployment model** — the persisted `alienation_calibration.joblib` is refitted on **all**
   labelled rows (standard practice: validate out-of-fold, deploy the full fit).

Known finding this measures rather than hides: the raw score is **non-monotonic** (the ~0.7
band is *less* often a true allegation than the ~0.45 band), so isotonic flattens that region —
calibration repairs the probability meaning at the cost of ranking granularity there.

In [ ]:
if ev is not None:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sklearn.isotonic import IsotonicRegression
    from sklearn.model_selection import StratifiedKFold
    import joblib

    raw = ev.alienation_conf.to_numpy(dtype=float)
    gold = ev.gold_alienation.to_numpy(dtype=float)
    N_BINS = 5
    bins = np.linspace(0, 1, N_BINS + 1)

    def bin_stats(conf, y):
        """Per-bin (mean confidence, observed positive rate, n) over equal-width bins."""
        idx = np.clip(np.digitize(conf, bins) - 1, 0, N_BINS - 1)
        out = []
        for b in range(N_BINS):
            m = idx == b
            if m.sum():
                out.append((conf[m].mean(), y[m].mean(), int(m.sum())))
        return out

    def ece(conf, y):
        """Expected Calibration Error: sum_b (n_b/N) * |obs_b - conf_b|."""
        return sum(n / len(y) * abs(o - c) for c, o, n in bin_stats(conf, y))

    # --- out-of-fold calibrated confidence (every row predicted by a fold that excluded it) ---
    oof = np.full_like(raw, np.nan)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for tr, te in skf.split(raw.reshape(-1, 1), gold):
        iso_cv = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
        iso_cv.fit(raw[tr], gold[tr])
        oof[te] = iso_cv.predict(raw[te])

    ece_raw, ece_oof = ece(raw, gold), ece(oof, gold)
    print(f"ECE raw            = {ece_raw:.3f}")
    print(f"ECE OOF-calibrated = {ece_oof:.3f}   <- the citable unseen-data calibration claim")

    # --- deployment model: refit on all labelled rows, persist ---
    iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
    iso.fit(raw, gold)
    joblib.dump(iso, CALIB_OUT)

    # --- reliability diagram: raw bins + OOF-calibrated bins + deployed isotonic curve ---
    rb, ob = bin_stats(raw, gold), bin_stats(oof, gold)
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect calibration")
    ax.scatter([x for x, _, _ in rb], [y for _, y, _ in rb],
               s=[20 + 6 * n for _, _, n in rb], color="#2c7fb8", zorder=3,
               label=f"raw bins (ECE={ece_raw:.3f})")
    ax.scatter([x for x, _, _ in ob], [y for _, y, _ in ob],
               s=[20 + 6 * n for _, _, n in ob], color="#33a02c", marker="s", zorder=3,
               label=f"OOF-calibrated bins (ECE={ece_oof:.3f})")
    grid = np.linspace(0, 1, 100)
    ax.plot(grid, iso.predict(grid), color="#d95f02", lw=2, label="deployed isotonic (fit on all)")
    for x, y, n in rb:
        ax.annotate(str(n), (x, y), textcoords="offset points", xytext=(4, 4), fontsize=8)
    ax.set_xlabel("confidence"); ax.set_ylabel("observed P(alleged=True)")
    ax.set_title("Reliability — alienation_alleged (5-fold OOF)")
    ax.legend(loc="upper left", fontsize=8)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    fig.tight_layout(); fig.savefig(FIG_OUT, dpi=130); plt.close(fig)
    print(f"wrote calibration -> {CALIB_OUT.name}")
    print(f"wrote reliability diagram -> {FIG_OUT.name}")
    print("raw bins (conf_mean, obs_rate, n):", [(round(x,2), round(y,2), n) for x, y, n in rb])
    print("OOF bins (conf_mean, obs_rate, n):", [(round(x,2), round(y,2), n) for x, y, n in ob])
    calib_metrics = dict(ece_raw=round(ece_raw, 3), ece_oof=round(ece_oof, 3))

ECE raw            = 0.179
ECE OOF-calibrated = 0.068   <- the citable unseen-data calibration claim
wrote calibration -> alienation_calibration.joblib
wrote reliability diagram -> alienation_reliability_diagram.png
raw bins (conf_mean, obs_rate, n): [(0.06, 0.06, 62), (0.3, 0.67, 12), (0.46, 0.82, 22), (0.72, 0.14, 14), (0.87, 0.8, 10)]
OOF bins (conf_mean, obs_rate, n): [(0.05, 0.07, 60), (0.56, 0.67, 24), (0.65, 0.53, 30), (0.86, 0.67, 6)]


## 6. Write the metrics report → `reports/`

In [ ]:
if ev is not None:
    lines = []
    lines.append("# ECHR extraction validation report\n")
    lines.append(f"- labelled sample: **{len(ev)}** cases, gold positive rate "
                 f"**{ev.gold_alienation.mean()*100:.1f}%**\n")
    lines.append("\n## Metadata tier — exact-match accuracy\n")
    for field, (acc, n) in meta_acc.items():
        lines.append(f"- `{field}`: **{acc:.3f}** (n={n})\n")
    lines.append("- caveat: `gold_outcome` was **pre-filled from the extractor's own guess** "
                 "and confirm-only during labelling, so this is a consistency check, **not an "
                 "independent validation** of the outcome field.\n")
    lines.append("\n## Content tier — alienation_alleged\n")
    lines.append(f"- precision **{content_metrics['precision']}**, recall "
                 f"**{content_metrics['recall']}**, F1 **{content_metrics['f1']}** "
                 f"(threshold {ALLEGED_THRESHOLD}, n={content_metrics['n']})\n")
    lines.append(f"- confusion: TP={content_metrics['tp']} FP={content_metrics['fp']} "
                 f"FN={content_metrics['fn']} TN={content_metrics['tn']}\n")
    lines.append("\n## Calibration (validated out-of-fold)\n")
    lines.append(f"- **ECE raw = {calib_metrics['ece_raw']}**, "
                 f"**ECE calibrated (5-fold out-of-fold) = {calib_metrics['ece_oof']}** — the "
                 f"calibrated figure is measured on rows the isotonic fit never saw, so it "
                 f"supports the unseen-data claim.\n")
    lines.append("- raw confidence is **non-monotonic** (the ~0.7 band is less often a true "
                 "allegation than the ~0.45 band); isotonic calibration flattens that region, "
                 "trading ranking granularity for probability meaning.\n")
    lines.append(f"- deployed isotonic mapping (refit on all labelled rows) persisted to "
                 f"`{CALIB_OUT.name}`; reliability diagram `{FIG_OUT.name}`.\n")
    lines.append("\n## Labelling protocol — limitations\n")
    lines.append("- Labels come from a **single annotator**; the template displayed the "
                 "extractor's guess and evidence sentence, so an **anchoring bias** cannot be "
                 "excluded (37/120 labels do overrule the extractor). No inter-annotator "
                 "agreement has been measured yet.\n")
    lines.append("\n## Domain of validity\n")
    lines.append("- Calibration holds **only within the validated domain: ECHR Article 8 "
                 "contact / parental-alienation cases**. It must be **re-run** to extend to "
                 "other Articles, other ECHR subject-matter, or other corpora (RIS/Swiss).\n")
    lines.append("- **Sample frame:** the frozen 120-case sample was stratified over the "
                 "2015\u20132025 corpus (719 cases). The corpus was extended to 2000\u20132025 "
                 "(1,116 cases) on 2026-07-07; the metrics above still validate the extractor "
                 "on those 120 cases, but pre-2015 judgments (older HUDOC formatting) are "
                 "**outside the sample frame** \u2014 re-stratifying and re-labelling over the "
                 "extended corpus is the clean fix.\n")
    REPORT_OUT.write_text("".join(lines), encoding="utf-8")
    print("wrote", REPORT_OUT.name)
    print("".join(lines))

wrote extraction_validation_report.md
# ECHR extraction validation report
- labelled sample: **120** cases, gold positive rate **33.3%**

## Metadata tier — exact-match accuracy
- `outcome`: **1.000** (n=95)
- caveat: `gold_outcome` was **pre-filled from the extractor's own guess** and confirm-only during labelling, so this is a consistency check, **not an independent validation** of the outcome field.

## Content tier — alienation_alleged
- precision **0.543**, recall **0.475**, F1 **0.507** (threshold 0.5, n=120)
- confusion: TP=19 FP=16 FN=21 TN=64

## Calibration (validated out-of-fold)
- **ECE raw = 0.179**, **ECE calibrated (5-fold out-of-fold) = 0.068** — the calibrated figure is measured on rows the isotonic fit never saw, so it supports the unseen-data claim.
- raw confidence is **non-monotonic** (the ~0.7 band is less often a true allegation than the ~0.45 band); isotonic calibration flattens that region, trading ranking granularity for probability meaning.
- deployed isotoni

## 7. Scope of validity (read me)
The fitted calibration is **domain-specific**. These numbers and the `raw → calibrated`
mapping are valid for **ECHR Article 8 contact / parental-alienation judgments** — the corpus
they were measured on. Applying this calibration to other Articles, other ECHR subject-matter,
or the RIS/Swiss corpora is **not** justified: re-freeze a labelled sample from the new domain
and **re-run this notebook**. Calibration is a claim about a distribution, not a universal
constant.